# `pairing.py` Reference

All pairing functions share the signature `(records, rng, ctx) → list[(a, b)]`.
They differ in how they order players within record groups.

`make_record_group_pairing` always builds **even-sized brackets** via
`standings.create_ranked_even_groups` before applying the within-group strategy,
so odd record groups are balanced by moving one player to the next bracket.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(
    Path.cwd().parent.parent
    if Path.cwd().name == 'reference'
    else Path.cwd().parent
))

from random import Random
print('ready')

ready


In [2]:
from tournament.generators import make_players, skilled_match, random_match
from tournament.engine import run_tournament
from tournament.pairing import get as get_pairing

rng = Random(42)
players = make_players(8, rng)
tour = run_tournament(players, n_rounds=4, pairing=get_pairing('adjacent'), rng=rng)

In [3]:
from tournament.standings import compute_records
records = compute_records(tour)

## `REGISTRY` and `get`

In [4]:
from tournament.pairing import REGISTRY, get as get_pairing

print('Registered strategies:', list(REGISTRY))

fn = get_pairing('fold')
print(f'get("fold") → {fn.__name__}')

try:
    get_pairing('unknown')
except KeyError as e:
    print(f'KeyError for unknown name: {e}')

Registered strategies: ['adjacent', 'fold', 'strong_weak', 'random_within_record', 'random']
get("fold") → record_group__fold
KeyError for unknown name: "unknown pairing 'unknown'; choose from ['adjacent', 'fold', 'strong_weak', 'random_within_record', 'random']"


## `PairingContext`

In [5]:
from tournament.pairing import PairingContext

ctx = PairingContext(
    past_opponents={p.pid: tour.past_opponents(p.pid) for p in players},
    round_number=5,
)
print(f'round_number: {ctx.round_number}')
print(f'past_opponents[0]: {ctx.past_opponents[0]}')

round_number: 5
past_opponents[0]: {1, 3, 4, 6}


## Strategy comparison — same records, different orderings

In [6]:
print(f'{"Strategy":<22}  pairings (name W-L)')
print('-' * 80)
for name in ('adjacent', 'fold', 'strong_weak', 'random_within_record'):
    pairs = get_pairing(name)(records, Random(0), ctx)
    row = '  '.join(
        f'{players[a].name}({records[a].wins}W) vs {players[b].name}({records[b].wins}W)'
        for a, b in pairs
    )
    print(f'{name:<22}  {row}')

Strategy                pairings (name W-L)
--------------------------------------------------------------------------------
adjacent                P003(6W) vs P004(5W)  P000(5W) vs P002(3W)  P001(4W) vs P007(3W)  P006(5W) vs P005(1W)
fold                    P003(6W) vs P004(5W)  P000(5W) vs P002(3W)  P001(4W) vs P007(3W)  P006(5W) vs P005(1W)
strong_weak             P003(6W) vs P006(5W)  P000(5W) vs P002(3W)  P001(4W) vs P007(3W)  P004(5W) vs P005(1W)
random_within_record    P004(5W) vs P003(6W)  P000(5W) vs P002(3W)  P001(4W) vs P007(3W)  P006(5W) vs P005(1W)


## `make_record_group_pairing` — factory with custom `rating_fn`

The factory always passes records through `standings.create_ranked_even_groups`,
so every bracket has an even number of players before the strategy is applied.

In [7]:
from tournament.pairing import make_record_group_pairing

# Rank within record groups by total agents scored (rather than differential)
custom_fn = make_record_group_pairing(
    'strong_weak',
    rating_fn=lambda seq: sum(f for f, _ in seq)
)
print('fn name:', custom_fn.__name__)

pairs = custom_fn(records, Random(0), ctx)
print('Pairs (strong_weak, total_agents_scored tiebreaker):')
for a, b in pairs:
    ra, rb = records[a], records[b]
    print(f'  {players[a].name} ({ra.wins}W diff={ra.agent_diff:+d}) '
          f'vs {players[b].name} ({rb.wins}W diff={rb.agent_diff:+d})')

fn name: record_group__strong_weak
Pairs (strong_weak, total_agents_scored tiebreaker):
  P003 (6W diff=+10) vs P006 (5W diff=+1)
  P000 (5W diff=+8) vs P002 (3W diff=-2)
  P001 (4W diff=-1) vs P007 (3W diff=-3)
  P004 (5W diff=+4) vs P005 (1W diff=-17)


## Rematch avoidance

In [8]:
# After 4 rounds with 8 players, rematches may be unavoidable.
# _avoid_rematches does a greedy swap pass.
print('Past opponents (showing potential rematches):')
for p in players:
    opp_names = [players[o].name for o in ctx.past_opponents[p.pid]]
    print(f'  {p.name}: already faced {opp_names}')

Past opponents (showing potential rematches):
  P000: already faced ['P001', 'P003', 'P004', 'P006']
  P001: already faced ['P000', 'P004', 'P005', 'P006']
  P002: already faced ['P003', 'P005', 'P006', 'P007']
  P003: already faced ['P000', 'P002', 'P005', 'P007']
  P004: already faced ['P000', 'P001', 'P005', 'P007']
  P005: already faced ['P001', 'P002', 'P003', 'P004']
  P006: already faced ['P000', 'P001', 'P002', 'P007']
  P007: already faced ['P002', 'P003', 'P004', 'P006']
